In [1]:
import yfinance as yf
import pandas as pd
import time
import os


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fix for:
    - MultiIndex columns
    - tuple columns
    - normal string columns
    """

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(i) for i in col]).lower() for col in df.columns]
    else:
        df.columns = [
            col.lower() if isinstance(col, str) else str(col).lower()
            for col in df.columns
        ]

    return df


def download_ticker_safe(ticker: str,
                          start="2010-01-01",
                          max_retries=5):

    for attempt in range(max_retries):
        try:
            df = yf.download(
                ticker,
                start=start,
                progress=False,
                threads=False,
                auto_adjust=False
            )

            if df is None or df.empty:
                raise ValueError("Empty dataframe")

            df = clean_columns(df)
            df = df.dropna()

            return df

        except Exception as e:
            print(f"[{ticker}] attempt {attempt+1}/{max_retries} failed: {e}")
            time.sleep(2 * (attempt + 1))

    print(f"[{ticker}] FAILED completely")
    return pd.DataFrame()


def save_df(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path)


if __name__ == "__main__":

    tickers = ["AAPL", "MSFT", "NVDA", "SPY"]

    for t in tickers:
        df = download_ticker_safe(t)

        if df.empty:
            print(f"{t}: empty, skipped")
            continue

        save_df(df, f"../../data/raw/{t}.csv")

        print(f"{t}: {df.shape}")

        time.sleep(1.5)

    print("DONE")

AAPL: (4114, 6)
MSFT: (4114, 6)
NVDA: (4114, 6)
SPY: (4114, 6)
DONE
